## Why work on computer vision models in 2026?

### -Every low quality image and video is low quality in a unique way. Upscaling degraded images requires custom degrading of images and custom training on what are small, local models


### -The CNN is still competitive with the transformer

### -Agents training in simulations and robots can save memory using a lightweight upscaling model

![Alt text](cnn.png)

![Alt text](unet.png)

In [2]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights

from IPython.display import clear_output
from collections import deque

import random
import time
import uuid
import math
import numpy as np
import io
from collections import OrderedDict
from PIL import Image
from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR

device = 'cuda:0'

C:\Users\ooo\AppData\Local\Temp\ipykernel_25424\3643537313.py:25: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [77]:
64*3

192

In [81]:
import io
import random
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode


class SimpleDegradation:
    """
    Clean PIL image in -> degraded PIL image out, ALWAYS at (out_size x out_size).

    Applied in this order:
        1. downscale
        2. chroma blur
        3. light Gaussian blur across all RGB channels
        4. jpeg compression
        5. final resize to out_size
        6. optional light Gaussian noise

    Tune everything in __init__. Set a probability to 0.0 to disable a stage.
    """

#     def __init__(self,    
#         out_size=512,
#         # Regular image blur
#         p_blur=0.75,
#         blur_min=.88,
#         blur_max=1.38,

#         p_resample=1.0,
#         scale_min=0.70,
#         scale_max=0.80,
#         # Chroma blur
#         p_chroma=0.5,
#         chroma_min=4.0,
#         chroma_max=4.0,

#         # JPEG
#         p_jpeg=0.7,
#         quality_min=70,
#         quality_max=84,
#         # Light noise
#         p_noise=2.0,
#         noise_max=0.004):

#     def __init__(self,
#          out_size=96,          # MUST match your x_size
#          # 1. downscale
#          # 2. regular image blur
#          p_blur=1,
#          blur_min=0.4,          # Gaussian sigma at the smaller resolution
#          blur_max=0.4,

#          p_resample=0,
#          scale_min=0.85,        # shrinks to 75%-77% of out_size
#          scale_max=0.87,
#          # 3. chroma (color) blur
#          p_chroma=0.1,
#          chroma_min=2,
#          chroma_max=2,

#          # 4. jpeg
#          p_jpeg=0,
#          quality_min=85,
#          quality_max=86,
#          # 5. light noise
#          p_noise=0,
#          noise_max=0.007):
        
        
        
#     def __init__(self,    
#          out_size=320,          # MUST match your x_size
#          # 1. downscale
#          # 2. regular image blur
#          p_blur=2,
#          blur_min=0.1,          # Gaussian sigma at the smaller resolution
#          blur_max=0.4,

#          p_resample=1.0,
#          scale_min=0.70,        # shrinks to 75%-77% of out_size
#          scale_max=0.88,
#          # 3. chroma (color) blur
#          p_chroma=0,
#          chroma_min=10,
#          chroma_max=2,

#          # 4. jpeg
#          p_jpeg=0.99,
#          quality_min=75,
#          quality_max=96,
#          # 5. light noise
#          p_noise=1.2,
#          noise_max=0.003):
        
        
    def __init__(self,
         out_size=192,
         # Regular image blur, before downscaling
         p_blur=5,
         blur_min=0.872,
         blur_max=0.9,

         p_resample=22,
         scale_min=0.96,
         scale_max=0.96,
         # Chroma blur, after downscaling
         p_chroma=0.1,
         chroma_min=1.0,
         chroma_max=5.0,

         # JPEG
         p_jpeg=10,
         quality_min=70,
         quality_max=70,
         # Light noise
         p_noise=1.0,
         noise_max=0.007):


        self.out_size = out_size
        self.p_resample, self.scale_min, self.scale_max = p_resample, scale_min, scale_max
        self.p_chroma, self.chroma_min, self.chroma_max = p_chroma, chroma_min, chroma_max
        self.p_blur, self.blur_min, self.blur_max = p_blur, blur_min, blur_max
        self.p_jpeg, self.quality_min, self.quality_max = p_jpeg, quality_min, quality_max
        self.p_noise, self.noise_max = p_noise, noise_max

    def __call__(self, img):
        img = img.convert("RGB")
        # ---- 3. REGULAR BLUR (softens brightness and color detail) ----
        if random.random() < self.p_blur:
            sigma = random.uniform(self.blur_min, self.blur_max)
            img = TF.gaussian_blur(img, kernel_size=[5, 5], sigma=[sigma, sigma])

        # ---- 1. DOWNSCALE (resolution loss) ----
        if random.random() < self.p_resample:
            scale = random.uniform(self.scale_min, self.scale_max)
            small = max(16, int(self.out_size * scale))
            mode = random.choice([InterpolationMode.BILINEAR, InterpolationMode.BICUBIC])
            img = TF.resize(img, (small, small), interpolation=mode)

        # ---- 2. CHROMA BLUR (color bleed) ----
        if random.random() < self.p_chroma:
            img = self._chroma_blur(img, random.uniform(self.chroma_min, self.chroma_max))


        # ---- 4. JPEG (blocking, done at the small size like real encoding) ----
        if random.random() < self.p_jpeg:
            q = random.randint(self.quality_min, self.quality_max)
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=q)
            buf.seek(0)
            img = Image.open(buf).convert("RGB")

        # ---- 5. RESIZE TO OUTPUT SIZE -- ALWAYS RUNS ----
        mode = random.choice([InterpolationMode.BILINEAR, InterpolationMode.BICUBIC])
        img = TF.resize(img, (self.out_size, self.out_size), interpolation=mode)

        # ---- 6. OPTIONAL LIGHT NOISE ----
        if random.random() < self.p_noise:
            x = TF.to_tensor(img)
            x = (x + torch.randn_like(x) * random.uniform(0.0, self.noise_max)).clamp(0, 1)
            img = TF.to_pil_image(x)

        return img

    @staticmethod
    def _chroma_blur(img, sigma):
        """Blur only the color channels (U,V), leave brightness (Y) sharp."""
        x = TF.to_tensor(img).unsqueeze(0)
        r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
        # RGB -> YUV
        y = 0.299 * r + 0.587 * g + 0.114 * b
        u = -0.14713 * r - 0.28886 * g + 0.436 * b
        v = 0.615 * r - 0.51499 * g - 0.10001 * b
        # Blur chroma only.
        k = max(3, int(sigma * 4) | 1)
        u = TF.gaussian_blur(u, kernel_size=[k, k], sigma=[sigma, sigma])
        v = TF.gaussian_blur(v, kernel_size=[k, k], sigma=[sigma, sigma])
        # YUV -> RGB
        r2 = y + 1.13983 * v
        g2 = y - 0.39465 * u - 0.58060 * v
        b2 = y + 2.03211 * u
        out = torch.cat([r2, g2, b2], dim=1).clamp(0, 1)
        return TF.to_pil_image(out.squeeze(0))


# ---------------------------------------------------------------------------
# usage: out_size MUST equal the x resolution your model trains on
# ---------------------------------------------------------------------------
degrade = SimpleDegradation(out_size=192)

def transform_x(pil_img):
    pil_img = degrade(pil_img)
    x = TF.to_tensor(pil_img)
    x = normalize(x)               # ImageNet normalize LAST
    return x

In [79]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

class RandomJpegCompression:
    def __init__(self, p=0.35, quality_min=65, quality_max=95):
        self.p, self.quality_min, self.quality_max = p, quality_min, quality_max

    def __call__(self, img):
        if random.random() > self.p:
            return img
        q = random.randint(self.quality_min, self.quality_max)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q, subsampling=0)
        buf.seek(0)
        return Image.open(buf).convert("RGB")


class RandomGaussianBlurX:
    def __init__(self, kmin=15, kmax=23, smin=1.5, smax=2.6):
        self.kmin, self.kmax = kmin, kmax
        self.smin, self.smax = smin, smax

    def __call__(self, img):
        k_lo = self.kmin if self.kmin % 2 == 1 else self.kmin + 1
        k_hi = self.kmax if self.kmax % 2 == 1 else self.kmax - 1
        if k_hi < k_lo:
            k_hi = k_lo
        k = random.randrange(k_lo, k_hi + 1, 2)
        sigma = random.uniform(self.smin, self.smax)
        return TF.gaussian_blur(img, kernel_size=[k, k], sigma=[sigma, sigma])


normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# transform_x = transforms.Compose([
#     RandomGaussianBlurX(),
#     transforms.ToTensor(),
#     normalize,
# ])

transform_clean = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])


def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=tensor.device).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)


def tensor_to_image(tensor):
    return Image.fromarray((tensor.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8'))


def apply_augmentation(x_img, y_img):
    if random.random() > 0.5:
        x_img, y_img = TF.hflip(x_img), TF.hflip(y_img)

    if random.random() < 0.35:
        x_img = TF.adjust_brightness(x_img, random.uniform(0.96, 1))
        x_img = TF.adjust_contrast(x_img, random.uniform(0.96, 1))
        x_img = TF.adjust_saturation(x_img, random.uniform(0.96, 1))
    x_img = RandomJpegCompression()(x_img)
    return x_img, y_img


def get_filenames_from_y(y_folder):
    exts = (".png", ".jpg", ".jpeg", ".webp")
    filenames = sorted(f for f in os.listdir(y_folder) if f.lower().endswith(exts))
    random.shuffle(filenames)
    return filenames


def split_filenames(filenames, train_ratio=0.96):
    n = int(len(filenames) * train_ratio)
    return filenames[:n], filenames[n:]

import torch.nn.functional as F

class SuperResDataset(Dataset):
    def __init__(self, y_folder, filenames):
        self.y_folder = y_folder
        self.filenames = filenames

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        # The CPU ONLY reads the file and converts it to a raw, normalized tensor.
        filename = self.filenames[idx]
        img = Image.open(os.path.join(self.y_folder, filename)).convert("RGB")
        
        # transform_clean = ToTensor() + Normalize()
        return transform_clean(img)

class DynamicGaussianBlur:
    """Scales the blur kernel automatically based on the image size."""
    def __init__(self, kernel_ratio=0.03, smin=1.4, smax=2.2):
        self.kernel_ratio = kernel_ratio # Kernel will be 3% of image width
        self.smin, self.smax = smin, smax

    def __call__(self, img):
        # Calculate dynamic kernel size
        w, h = img.size
        k = int(min(w, h) * self.kernel_ratio)
        k = k if k % 2 == 1 else k + 1 # Ensure odd number
        k = max(3, k) # Never go below a 3x3 kernel
        
        sigma = random.uniform(self.smin, self.smax)
        return TF.gaussian_blur(img, kernel_size=[k, k], sigma=[sigma, sigma])

# transform_x = transforms.Compose([
#     DynamicGaussianBlur(kernel_ratio=0.03), # Safe at any resolution
#     transforms.ToTensor(),
#     normalize,
# ])

        
def load_images_from_filenames(y_folder, filenames, batch_size,
                                x_size=512, y_size=1024, augment=False):
    """Full training loader. X = degraded at x_size, Y = clean at y_size."""
    x_images, y_images = [], []
    for filename in filenames:
        y_img = Image.open(os.path.join(y_folder, filename)).convert("RGB")
#         y_img = TF.resize(y_img, (y_size, y_size), interpolation=InterpolationMode.BICUBIC)
        x_img = TF.resize(y_img, (x_size, x_size), interpolation=InterpolationMode.BICUBIC)

        if augment:
            x_img, y_img = apply_augmentation(x_img, y_img)

        x_tensor = transform_x(x_img)
        y_tensor = transform_clean(y_img)

        if augment and random.random() < 0.35:
            x_tensor = x_tensor + torch.randn_like(x_tensor) * random.uniform(0.0, 0.02)

        x_images.append(x_tensor)
        y_images.append(y_tensor)

        if len(x_images) == batch_size:
            yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()
            x_images, y_images = [], []
    if x_images:
        yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()


def load_pretrain_images(y_folder, filenames, batch_size, x_size=64, y_size=128, augment=False):
    """
    PROPER PRETRAIN LOADER:
    Extracts a native y_size crop from the full image, then downscales it to x_size.
    Preserves actual textures without scale-warping.
    """
    x_images, y_images = [], []
    for filename in filenames:
        img = Image.open(os.path.join(y_folder, filename)).convert("RGB")
        
        # 1. THE FIX: Crop a true high-res patch from the original image
        # This gives you a native 128x128 piece of the actual 4K texture
        i, j, h, w = transforms.RandomCrop.get_params(img, output_size=(y_size, y_size))
        y_img = TF.crop(img, i, j, h, w)
        
        # 2. Downscale the high-res crop to create the degraded input
        x_img = TF.resize(y_img, (x_size, x_size), interpolation=InterpolationMode.BICUBIC)

        if augment:
            # We skip JPEG compression here if the image is too small (e.g., < 128)
            # to avoid destroying the 8x8 block grids.
            if x_size >= 128:
                x_img = RandomJpegCompression()(x_img)
            
            x_img, y_img = apply_augmentation(x_img, y_img)

        # Dynamic Blur is handled inside transform_x now
        x_tensor = transform_x(x_img)
        y_tensor = transform_clean(y_img)

        if augment and random.random() < 0.35:
            # Scale the noise down slightly for smaller images to avoid burying the signal
            noise_scale = 0.01 if x_size <= 64 else 0.02
            x_tensor = x_tensor + torch.randn_like(x_tensor) * random.uniform(0.0, noise_scale)

        x_images.append(x_tensor)
        y_images.append(y_tensor)

        if len(x_images) == batch_size:
            yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()
            x_images, y_images = [], []
            
    if x_images:
        yield torch.stack(x_images).cuda(), torch.stack(y_images).cuda()

In [13]:
# ============================================================================
#  mHC CORE
# ============================================================================

class RMSNorm2d(nn.Module):
    """RMSNorm for (B, C, H, W) feature maps. Paper specifies RMSNorm, not LayerNorm."""
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        x_float = x.float()
        rms = torch.sqrt(x_float.pow(2).mean(dim=1, keepdim=True) + self.eps)
        return (x_float / rms).to(x.dtype) * self.scale


def sinkhorn_knopp(mat, num_iters=10):
    """
    Project onto the Birkhoff polytope (doubly stochastic matrices).
    mat: (..., n, n) raw logits → exp → iterative row/col normalization.
    """
    m = torch.exp(mat.float())
    for _ in range(num_iters):
        m = m / (m.sum(dim=-1, keepdim=True) + 1e-8)
        m = m / (m.sum(dim=-2, keepdim=True) + 1e-8)
    return m.to(mat.dtype)


class mHCUnit(nn.Module):
    """
    Core mHC mechanism wrapping a single shape-preserving layer.

    Given widened stream (B, n*C, H, W):
      1. H_pre aggregates n streams → layer input (B, C, H, W)
      2. layer_fn processes it
      3. H_post broadcasts output back to n streams
      4. H_res mixes the residual streams (doubly stochastic)
      5. New stream = H_res @ old_stream + H_post * layer_output
    """
    def __init__(self, dim, expansion=4, sinkhorn_iters=5):
        super().__init__()

        # dim = number of channels per stream.  Example: dim=64
        self.dim = dim

        # n = number of parallel streams.  Example: n=4
        # So the total "widened" tensor is n*dim = 256 channels.
        self.n = expansion

        # stream_dim = total channels in the widened stream = n * dim = 256
        self.stream_dim = dim * expansion

        # How many times to iterate Sinkhorn-Knopp (more = closer to true
        # doubly stochastic, 10 is plenty for a 4x4 matrix).
        self.sinkhorn_iters = sinkhorn_iters

        # =====================================================================
        # STATIC BIASES — fixed starting values for the three coefficient sets.
        # These define the "default" behavior before the network learns anything.
        # They are nn.Parameters so backprop can adjust them during training.
        # =====================================================================

        # b_pre: shape (n,) = (4,) — one bias value per stream.
        #
        # These biases feed into softmax to produce the 4 weights that collapse
        # the 4 streams into a single layer input. We want stream[0] (the main
        # signal) to dominate at the start, so we set its bias high (2.0) and
        # the rest low (-1.0).
        #
        # softmax([2.0, -1.0, -1.0, -1.0]) ≈ [0.95, 0.017, 0.017, 0.017]
        # → at init, the layer input is ~95% stream[0], barely any from others.
        b_pre_init = torch.full((expansion,), -1.0)
        b_pre_init[0] = 2.0
        self.b_pre = nn.Parameter(b_pre_init.clone())

        # b_post: shape (n,) = (4,) — one bias value per stream.
        #
        # These biases feed into sigmoid to produce per-stream gates that
        # control how much of the layer's output each stream receives.
        # Same init values, but sigmoid interprets them differently:
        #   sigmoid(2.0) = 0.88  → stream[0] receives 88% of the layer output
        #   sigmoid(-1.0) = 0.27 → streams[1-3] receive 27% each
        self.b_post = nn.Parameter(b_pre_init.clone())

        # b_res: shape (n, n) = (4, 4) — a full mixing matrix.
        #
        # WHY 4×4? Because this matrix defines how each of the 4 NEW streams
        # is built from the 4 OLD streams:
        #   new_stream[0] = m[0,0]*old[0] + m[0,1]*old[1] + m[0,2]*old[2] + m[0,3]*old[3]
        #   new_stream[1] = m[1,0]*old[0] + m[1,1]*old[1] + m[1,2]*old[2] + m[1,3]*old[3]
        #   ...
        # Row i defines where new_stream[i] comes from. Column j defines how
        # much old_stream[j] contributes to all new streams.
        #
        # We want this to start as near-identity (each stream keeps itself):
        #   [[ 5, -3, -3, -3],     after Sinkhorn     [[~0.97, ~0.01, ~0.01, ~0.01],
        #    [-3,  5, -3, -3],     ──────────────→      [~0.01, ~0.97, ~0.01, ~0.01],
        #    [-3, -3,  5, -3],                          [~0.01, ~0.01, ~0.97, ~0.01],
        #    [-3, -3, -3,  5]]                          [~0.01, ~0.01, ~0.01, ~0.97]]
        b_res_init = torch.full((expansion, expansion), -3.0)
        b_res_init.fill_diagonal_(5.0)
        self.b_res = nn.Parameter(b_res_init)

        # =====================================================================
        # GATING SCALARS (alpha) — control how much the dynamic (data-dependent)
        # part contributes vs the static biases above.
        #
        # At init, alpha = 0.01, so the dynamic part is multiplied by 0.01,
        # making it negligible. The static biases above dominate completely.
        # As training progresses, alpha grows and the network starts using
        # data-dependent coefficients.
        # =====================================================================
        self.alpha_pre = nn.Parameter(torch.tensor(0.01))
        self.alpha_post = nn.Parameter(torch.tensor(0.01))
        self.alpha_res = nn.Parameter(torch.tensor(0.01))

        # =====================================================================
        # DYNAMIC PROJECTIONS (phi) — 1x1 convolutions that look at the current
        # stream content and produce per-pixel coefficients.
        #
        # Think of these as: "look at what's in the streams RIGHT NOW and decide
        # how to mix them for THIS specific pixel." Different pixels can get
        # different mixing coefficients.
        #
        # phi_pre:  input = 256ch stream → output = 4 values (one weight per stream)
        # phi_post: input = 256ch stream → output = 4 values (one gate per stream)
        # phi_res:  input = 256ch stream → output = 16 values (a full 4×4 matrix)
        #
        # phi_res outputs 4*4=16 because it needs to produce a complete mixing
        # matrix. phi_pre and phi_post only output 4 because they are vectors,
        # not matrices.
        # =====================================================================
        self.phi_pre = nn.Conv2d(self.stream_dim, expansion, 1, bias=False)
        self.phi_post = nn.Conv2d(self.stream_dim, expansion, 1, bias=False)
        self.phi_res = nn.Conv2d(self.stream_dim, expansion * expansion, 1, bias=False)

        # RMSNorm on the full 256ch stream — normalizes the stream before
        # feeding it into the phi convolutions, for training stability.
        self.norm = RMSNorm2d(self.stream_dim)

        # Initialize all phi convolution weights to ZERO.
        # This means at the start of training:
        #   phi_pre(stream) = 0, phi_post(stream) = 0, phi_res(stream) = 0
        # So the raw coefficients = alpha * 0 + static_bias = just the static bias.
        # The network starts with pure static behavior (identity-like) and
        # gradually learns to use the dynamic data-dependent part.
        nn.init.zeros_(self.phi_pre.weight)
        nn.init.zeros_(self.phi_post.weight)
        nn.init.zeros_(self.phi_res.weight)
        
#         self.output_gate = nn.Parameter(torch.zeros(1))
        self.output_gate = nn.Parameter(torch.ones(1))   # was torch.zeros(1)
    def forward(self, stream, layer_fn):
        """
        stream: (B, n*C, H, W) — the widened multi-stream tensor
        layer_fn: callable (B, C, H, W) → (B, C, H, W) — e.g. a Conv3x3 block
        Returns: (B, n*C, H, W) — updated multi-stream tensor
        """
        B, nC, H, W = stream.shape
        C, n = self.dim, self.n
        # Example: B=2, C=64, n=4, nC=256, H=W=44

        # =====================================================================
        # STEP 0: Normalize the stream, then compute the three sets of
        # coefficients. Each set has a STATIC part (the bias, same for every
        # pixel) and a DYNAMIC part (produced by a 1x1 conv, different per pixel).
        # The formula for each is:  raw = alpha * dynamic + bias
        # =====================================================================

        # Normalize the 256ch stream for stable coefficient computation
        s_norm = self.norm(stream)

        # Run the three 1x1 convolutions on the normalized stream.
        # These look at what's currently in ALL 4 streams and produce
        # data-dependent coefficients for this specific pixel.
        #
        # phi_pre:  256ch → 4 values per pixel (one weight per stream)
        h_pre_dyn = self.phi_pre(s_norm)                         # (B, 4, H, W)
        # phi_post: 256ch → 4 values per pixel (one gate per stream)
        h_post_dyn = self.phi_post(s_norm)                       # (B, 4, H, W)
        # phi_res:  256ch → 16 values per pixel → reshaped into a 4×4 matrix
        h_res_dyn = self.phi_res(s_norm).view(B, n, n, H, W)    # (B, 4, 4, H, W)

        # Combine: raw = (tiny alpha) * (data-dependent) + (static bias)
        # At init: alpha=0.01, phi outputs 0 → raw = 0 + bias = just the bias.
        # Over training: alpha grows, phi learns → raw becomes data-dependent.
        #
        # b_pre is shape (4,), we reshape to (1, 4, 1, 1) so it broadcasts
        # across batch, height, and width dimensions.
        h_pre_raw = self.alpha_pre * h_pre_dyn + self.b_pre.view(1, n, 1, 1)
        h_post_raw = self.alpha_post * h_post_dyn + self.b_post.view(1, n, 1, 1)
        # b_res is shape (4, 4), reshaped to (1, 4, 4, 1, 1) for broadcasting.
        h_res_raw = self.alpha_res * h_res_dyn + self.b_res.view(1, n, n, 1, 1)

        # =====================================================================
        # STEP 1: Apply manifold constraints to turn raw numbers into valid
        # coefficients that can't cause signal explosion.
        # =====================================================================

        # H_pre: softmax across the n=4 streams → weights that sum to 1.
        # At init: softmax([2, -1, -1, -1]) ≈ [0.95, 0.017, 0.017, 0.017]
        # This means the layer input will be ~95% stream[0].
        H_pre = torch.softmax(h_pre_raw, dim=1)  # (B, 4, H, W)

        # H_post: sigmoid on each value independently → each gate ∈ (0, 1).
#         At init: sigmoid(2)=0.88 for stream[0], sigmoid(-1)=0.27 for others.
        H_post = torch.sigmoid(h_post_raw)        # (B, 4, H, W)

        # H_res: Sinkhorn-Knopp projection → doubly stochastic 4×4 matrix.
        # We need to reshape so each pixel's 4×4 matrix is in the last two dims.
        # (B, 4, 4, H, W) → permute → (B, H, W, 4, 4) → flatten → (B*H*W, 4, 4)
        h_sk = h_res_raw.permute(0, 3, 4, 1, 2).reshape(-1, n, n)  # (B*H*W, 4, 4)
        # Run Sinkhorn: exp() then alternate row/column normalization 10 times.
        # Result: every row sums to 1, every column sums to 1 → can't explode.
        H_res = sinkhorn_knopp(h_sk, self.sinkhorn_iters)
        # Reshape back: (B*H*W, 4, 4) → (B, H, W, 4, 4) → permute → (B, 4, 4, H, W)
        H_res = H_res.view(B, H, W, n, n).permute(0, 3, 4, 1, 2)  # (B, 4, 4, H, W)

        # =====================================================================
        # STEP 2: Use the coefficients to process the streams.
        # =====================================================================

        # Reshape stream from (B, 256, H, W) into (B, 4, 64, H, W)
        # so we can work with individual streams.
        stream_r = stream.view(B, n, C, H, W)

        # --- H_pre: collapse 4 streams into 1 layer input ---
        #
        # H_pre is (B, 4, H, W) — 4 weights per pixel.
        # stream_r is (B, 4, 64, H, W) — 4 streams of 64 channels.
        #
        # unsqueeze(2) adds a channel dim: H_pre becomes (B, 4, 1, H, W)
        # Multiply: (B, 4, 1, H, W) * (B, 4, 64, H, W) = (B, 4, 64, H, W)
        #   → each stream's 64 channels are scaled by that stream's weight
        # .sum(dim=1): add up across the 4 streams → (B, 64, H, W)
        #
        # This is a weighted average: input = w0*stream[0] + w1*stream[1] + ...
        layer_input = (H_pre.unsqueeze(2) * stream_r).sum(dim=1)  # (B, 64, H, W)

        # --- Run the actual layer (e.g. Conv3x3 + GroupNorm + LeakyReLU) ---
        # Only ONE 64-channel tensor goes in, ONE 64-channel tensor comes out.
        layer_output = layer_fn(layer_input)  # (B, 64, H, W)

        # --- H_post: distribute layer output to all 4 streams ---
        #
        # H_post is (B, 4, H, W) — one gate per stream.
        # layer_output is (B, 64, H, W).
        #
        # unsqueeze(2): H_post becomes (B, 4, 1, H, W)
        # unsqueeze(1): layer_output becomes (B, 1, 64, H, W)
        # Multiply: (B, 4, 1, H, W) * (B, 1, 64, H, W) = (B, 4, 64, H, W)
        #   → the same layer output is copied to all 4 streams, each scaled
        #     by its own gate value (0.88 for stream[0], 0.27 for others at init)
        stream_update = H_post.unsqueeze(2) * layer_output.unsqueeze(1)  # (B, 4, 64, H, W)

        # --- H_res: mix old streams via the 4×4 doubly stochastic matrix ---
        #
        # For each pixel, this does a matrix multiply:
        #   mixed_stream[i] = Σ_j  H_res[i,j] * old_stream[j]
        #
        # einsum letters: b=batch, i=output_stream, j=input_stream,
        #                 c=channel, h=height, w=width
        # j appears in both inputs but not output → summed over (matrix multiply)
        #
        # At init H_res ≈ identity, so mixed ≈ old streams unchanged.
        mixed = torch.einsum('bijhw,bjchw->bichw', H_res, stream_r)  # (B, 4, 64, H, W)

        # --- Final: add the residual-mixed streams + the layer output ---
        #
        # new_stream = (H_res @ old_streams) + (H_post * layer_output)
        #            = mixed old streams     + gated new information
        #
        # At init this ≈ old_streams + small * layer_output, which is basically
        # a residual connection. Over training, the mixing becomes learned.
        new_stream = mixed + (self.output_gate * stream_update)
        return new_stream.reshape(B, nC, H, W)

In [14]:
class mHCStage(nn.Module):
    """Two updates to the same streams: spatial convolution, then refinement."""
    def __init__(self, channels, expansion, sinkhorn_iters):
        super().__init__()
        self.mhc_conv = mHCUnit(channels, expansion, sinkhorn_iters)
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, padding_mode='reflect', bias=False),
            GN(channels), nn.LeakyReLU(0.1, inplace=True))

        self.mhc_ffn = mHCUnit(channels, expansion, sinkhorn_iters)
        self.ffn = nn.Sequential(
            nn.Conv2d(channels, channels * 2, 3, padding=1, padding_mode='reflect', bias=False),
            GN(channels * 2), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels * 2, channels, 3, padding=1, padding_mode='reflect', bias=False),
            GN(channels))
        self.se = SqueezeExcitation(channels, se_ratio=0.25)

    def refine(self, x):
        return self.se(self.ffn(x))

    def forward(self, stream):
        stream = self.mhc_conv(stream, self.conv)
        stream = self.mhc_ffn(stream, self.refine)
        return stream

class mHCMiddleBlock(nn.Module):
    """Convolutional U-Net bottleneck. Input and output: (B, channels, H, W)."""
    def __init__(self, channels=512, expansion=4, sinkhorn_iters=5, depth=2):
        super().__init__()
        if min(channels, expansion, sinkhorn_iters, depth) < 1:
            raise ValueError("channels, expansion, sinkhorn_iters and depth must be positive")
        self.channels = channels
        self.expansion = expansion
        self.stream_init = nn.Conv2d(channels, channels * expansion, 1, bias=False)
        self.stream_out = nn.Conv2d(channels * expansion, channels, 1, bias=False)

        # Rectangular identity: start by writing to and reading from stream 0.
        nn.init.dirac_(self.stream_init.weight)
        nn.init.dirac_(self.stream_out.weight)

        self.stages = nn.ModuleList([
            mHCStage(channels, expansion, sinkhorn_iters) for _ in range(depth)
        ])

    def forward(self, x):
        stream = self.stream_init(x)
        for stage in self.stages:
            stream = stage(stream)
        return self.stream_out(stream)

In [15]:
class EfficientNetB1SingleEncoder(nn.Module):
    def __init__(self, weights=EfficientNet_B1_Weights.DEFAULT):
        super().__init__()
        base_model = efficientnet_b1(weights=weights)
        self.features = base_model.features[:-1]

    def forward(self, x):
        chosen_indices = [1, 2, 3, 5, 7]
        temp_feats = []
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in chosen_indices:
                temp_feats.append(x)
        return temp_feats

In [16]:
class EfficientNetB5SingleEncoder(nn.Module):
    def __init__(self, weights=EfficientNet_B5_Weights.DEFAULT):
        super().__init__()
        base_model = efficientnet_b5(weights=weights)
        self.features = base_model.features[:-1]

    def forward(self, x):
        chosen_indices = [1, 2, 3, 5, 7]
        temp_feats = []
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in chosen_indices:
                temp_feats.append(x)
        return temp_feats

In [17]:
context_encoder = EfficientNetB1SingleEncoder().to(device)
for p in context_encoder.parameters():
    p.requires_grad = False
context_encoder.eval()

context_encoder_b5 = EfficientNetB5SingleEncoder().to(device)
for p in context_encoder_b5.parameters():
    p.requires_grad = False
context_encoder_b5.eval()

EfficientNetB5SingleEncoder(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
            (1): BatchNorm2d(48, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2)

In [70]:
class Top1MoE(nn.Module):
    def __init__(self, ch, num_experts=3, router_noise=0.1,
                 num_steps=2, num_streams=2, res_scale=0.1,
                 sinkhorn_iters=20):
        super().__init__()

        if ch < 1 or num_experts < 1 or num_steps < 1:
            raise ValueError("ch, num_experts, and num_steps must be positive")
        if num_streams < 2:
            raise ValueError("Use at least two streams for mHC")
        if sinkhorn_iters < 1 or router_noise < 0 or res_scale < 0:
            raise ValueError("Invalid Sinkhorn, noise, or residual-scale setting")

        self.ch = ch
        self.num_steps = num_steps
        self.num_streams = num_streams
        self.res_scale = res_scale
        self.num_experts = num_experts
        self.router_noise = router_noise

        self.router = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, num_experts),
        )

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(ch, ch, 3, padding=1),
                nn.GELU(),
                nn.Conv2d(ch, ch, 3, padding=1),
            )
            for _ in range(num_experts)
        ])

        # Each expert owns one mHC unit.
        # Both the expert and its mHC unit are reused across repetitions.
        self.mhc = nn.ModuleList([
            mHCUnit(
                dim=ch,
                expansion=num_streams,
                sinkhorn_iters=sinkhorn_iters,
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.ch or x.shape[0] == 0:
            raise ValueError(f"Expected nonempty (B, {self.ch}, H, W) input")
        if len(self.experts) != self.num_experts:
            raise RuntimeError("The expert count must match num_experts")

        # Keep routing calculations in FP32 under autocast.
        with torch.autocast(device_type=x.device.type, enabled=False):
            scores = self.router(x.float())

            if self.training and self.router_noise > 0:
                scores = scores + self.router_noise * torch.randn_like(scores)

            probabilities = scores.softmax(dim=1)
            selected_weights, selected_experts = probabilities.max(dim=1)

        output = torch.zeros_like(x)

        for expert_id, expert in enumerate(self.experts):
            image_ids = torch.where(selected_experts == expert_id)[0]

            if image_ids.numel() == 0:
                continue

            expert_input = x.index_select(0, image_ids)
            batch, channels, height, width = expert_input.shape

            # Start each stream with a copy of the selected images' features.
            # Packed shape: (batch, num_streams * channels, height, width).
            stream = expert_input.repeat(1, self.num_streams, 1, 1)
            mhc_unit = self.mhc[expert_id]

            # mHC calls this once per repetition.
            # Scale only the expert correction, not the carried streams.
            def scaled_expert(z):
                return self.res_scale * expert(z)

            for _ in range(self.num_steps):
                stream = mhc_unit(stream, scaled_expert)

            # Combine streams only after all repetitions have finished.
            state = stream.reshape(
                batch, self.num_streams, channels, height, width
            ).mean(dim=1)

            # Return a correction because DecoderBlock adds its own residual.
            expert_output = state - expert_input

            gates = selected_weights.index_select(
                0, image_ids
            ).reshape(-1, 1, 1, 1)

            straight_through_gate = 1.0 + (gates - gates.detach())
            weighted_output = (
                expert_output
                * straight_through_gate.to(expert_output.dtype)
            )

            output = output.index_copy(
                0, image_ids, weighted_output.to(output.dtype)
            )

        usage = F.one_hot(
            selected_experts, self.num_experts
        ).float().mean(dim=0)

        mean_probability = probabilities.mean(dim=0)
        balance_loss = (
            self.num_experts * (usage * mean_probability).sum()
        )

        return output, balance_loss

In [21]:
tm = Top1MoE(3).to('cuda')
r = torch.rand(30, 3, 128, 128).cuda()
o, _ = tm(r)

In [22]:
o.shape

torch.Size([30, 3, 128, 128])

In [71]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, se_ratio=0.25,
        upsample_scale=2, blur=False, num_experts=1, router_noise=0.1, num_steps=2):
        super().__init__()
        self.contract = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 2, 1, bias=False),
            GN(in_ch // 2),
            nn.LeakyReLU(0.1, inplace=True))
        self.upsample = PixelShuffle_ICNR(
            in_ch // 2, in_ch // 2,
            scale=upsample_scale, blur=blur)
        self.expand = nn.Sequential(
            nn.Conv2d(in_ch // 2, out_ch, 1, bias=False),
            GN(out_ch),
            nn.LeakyReLU(0.1, inplace=True))

        self.expert_refinement = Top1MoE(
            ch=out_ch, num_experts=num_experts, router_noise=router_noise, num_steps=num_steps)

        self.expert_refinement.experts = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    out_ch, out_ch, 3, padding=1, padding_mode="reflect", bias=False),
                GN(out_ch),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(
                    out_ch, out_ch, 3, padding=1, padding_mode="reflect", bias=False),
                GN(out_ch),
                nn.LeakyReLU(0.1, inplace=True),)
            for _ in range(num_experts)
        ])

        self.se = SqueezeExcitation(out_ch, se_ratio)
        self.skip_conv = nn.Conv2d(in_ch, out_ch, 1, bias=False)

        # Two streams: decoder features and the projected outer skip.
        self.skip_mhc = mHCUnit(
            dim=out_ch, expansion=2, sinkhorn_iters=20)

        # mHC reads the streams into out_ch channels before this operation.
        self.skip_refine = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.LeakyReLU(0.1, inplace=True))

        # Learn how to collapse the two updated streams.
        self.skip_out = nn.Conv2d(out_ch * 2, out_ch, 1, bias=False)

        with torch.no_grad():
            # Initially suppress the new correction; this gate remains trainable.
            self.skip_mhc.output_gate.zero_()

            # Initialize collapse as stream_0 + stream_1.
            identity = torch.eye(
                out_ch,
                device=self.skip_out.weight.device,
                dtype=self.skip_out.weight.dtype)
            self.skip_out.weight.zero_()
            self.skip_out.weight[:, :out_ch, 0, 0].copy_(identity)
            self.skip_out.weight[:, out_ch:, 0, 0].copy_(identity)

    def forward(self, x):
        skip = self.skip_conv(x)
        x = self.expand(self.upsample(self.contract(x)))
        res = x

        x, balance_loss = self.expert_refinement(x)
        x = self.se(x) + res

        skip = F.interpolate(
            skip, size=x.shape[2:],
            mode="bilinear", align_corners=False
        )

        # Pack two out_ch-channel streams into one 2*out_ch-channel tensor.
        stream = torch.cat([x, skip], dim=1)
        stream = self.skip_mhc(stream, self.skip_refine)
        x = self.skip_out(stream)

        return x, balance_loss

# U

In [72]:
class EncoderSkipBlock(DecoderBlock):
    """Your original DecoderBlock with a learned mHC correction at its input."""
    def __init__(self, decoder_ch, encoder_ch, out_ch, *, skip_width=16,
                 skip_iters=5, **block_args):
        super().__init__(decoder_ch + encoder_ch, out_ch, **block_args)
        if skip_width < 1 or skip_iters < 1:
            raise ValueError("skip_width and skip_iters must be positive")
        width = min(skip_width, decoder_ch, encoder_ch)
        self.from_decoder = nn.Conv2d(decoder_ch, width, 1, bias=False)
        self.from_encoder = nn.Conv2d(encoder_ch, width, 1, bias=False)
        self.encoder_mhc = mHCUnit(dim=width, expansion=2, sinkhorn_iters=skip_iters)
        self.skip_update = nn.Sequential(nn.Conv2d(width, width, 1, bias=False),
                                         nn.LeakyReLU(0.1))
        self.to_concat = nn.Conv2d(2 * width, decoder_ch + encoder_ch, 1, bias=False)
        nn.init.zeros_(self.to_concat.weight)

    def forward(self, x, encoder_skip):
        # Same raw features and channel order as your original torch.cat.
        original = torch.cat((x, encoder_skip), dim=1)
        streams = torch.cat((self.from_decoder(x), self.from_encoder(encoder_skip)), dim=1)
        streams = self.encoder_mhc(streams, self.skip_update)
        return super().forward(original + self.to_concat(streams))

class TwoStreamReadout(nn.Module):
    """One local mHC junction. Returns a learned adjustment, initially zero."""
    def __init__(self, left_ch, right_ch, width, out_ch):
        super().__init__()
        self.left = nn.Conv2d(left_ch, width, 1, bias=False)
        self.right = nn.Conv2d(right_ch, width, 1, bias=False)
        self.mix = mHCUnit(dim=width, expansion=2, sinkhorn_iters=5)
        self.update = nn.Sequential(nn.Conv2d(width, width, 1, bias=False),
                                    nn.LeakyReLU(0.1))
        self.readout = nn.Conv2d(2 * width, out_ch, 1, bias=False)
        nn.init.zeros_(self.readout.weight)

    def forward(self, left, right):
        streams = torch.cat((self.left(left), self.right(right)), dim=1)
        return self.readout(self.mix(streams, self.update))

class UNet(nn.Module):
    def __init__(self, num_classes=3, expansion=4, in_ch=3,
                 head_ch=64, n_stem_blocks=6, n_head_blocks=3, skip_width=16,
                 fuse_mhc=True, output_mhc=True):
        super().__init__()
        if num_classes != in_ch:
            raise ValueError("The bicubic residual requires num_classes == in_ch.")

        self.context_encoder = EfficientNetB1SingleEncoder().requires_grad_(False).eval()
        self.context_encoder_b5 = EfficientNetB5SingleEncoder().requires_grad_(False).eval()

        def skip(dec, enc, out, **kwargs):
            return EncoderSkipBlock(dec, enc, out, skip_width=skip_width, blur=False, **kwargs)

        self.refine_16_0 = DecoderBlock(512, 512, upsample_scale=1, num_experts=4, blur=False)
        self.refine_16_0_norm = GN(512)
        self.middle = mHCMiddleBlock(channels=512, expansion=expansion)
        self.d0_norm = GN(512)

        # Arguments to skip: decoder channels, encoder channels, output channels.
        self.refine_16_1 = skip(512, 512, 320, upsample_scale=1, num_experts=6)
        self.refine_16_1_norm = GN(320)
        self.d1 = skip(320, 320, 176)
        self.d1_norm = GN(176)
        self.refine_32 = skip(176, 176, 112, upsample_scale=1, num_experts=4)
        self.refine_32_norm = GN(112)
        self.d2 = skip(112, 112, 64)
        self.d2_norm = GN(64)
        self.refine_64 = skip(64, 64, 40, upsample_scale=1, num_experts=4)
        self.refine_64_norm = GN(40)
        self.d3 = skip(40, 40, 80, num_experts=3)
        self.d3_norm = GN(80)
        self.refine_40 = skip(80, 40, 80, upsample_scale=1, num_experts=2)
        self.refine_40_norm = GN(80)
        self.d4 = skip(80, 24, 80, num_experts=2)
        self.d4_norm = GN(80)
        self.refine_24 = skip(80, 24, 48, upsample_scale=1, num_experts=2)
        self.refine_24_norm = GN(48)
        self.d5 = skip(48, 16, head_ch, num_experts=2)
        self.d5_norm = GN(head_ch)

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, head_ch, 3, padding=1, padding_mode='reflect'),
            nn.LeakyReLU(0.1, inplace=True),
            *[ResBlockNoNorm(head_ch) for _ in range(n_stem_blocks)])
        self.fuse = nn.Conv2d(head_ch * 2, head_ch, 1, bias=False)
        self.head = nn.Sequential(*[EdgeResBlock(head_ch) for _ in range(n_head_blocks)])
        self.to_rgb = nn.Conv2d(head_ch, num_classes * 4, 3, padding=1, padding_mode='reflect')
        self.shuffle = nn.PixelShuffle(2)
        # Separate switches for training/timing the two experiments.
        self.fuse_mhc = (TwoStreamReadout(head_ch, head_ch, 8, head_ch)
                         if fuse_mhc else None)
        # RGB input and predicted RGB subpixels are already on the same LR grid.
        self.output_mhc = (TwoStreamReadout(in_ch, num_classes * 4, 4, 4)
                           if output_mhc else None)
        nn.init.zeros_(self.to_rgb.weight)
        nn.init.zeros_(self.to_rgb.bias)

    def train(self, mode=True):
        super().train(mode)
        self.context_encoder.eval()
        self.context_encoder_b5.eval()
        return self

    def _run_block(self, block, norm, losses, *inputs):
        features, aux = block(*inputs)
        losses.append(aux)
        return norm(features)

    def forward(self, x):
        initial_x = x
        b1 = self.context_encoder(x)
        b5 = self.context_encoder_b5(x)
        balance_losses = []

        x = self._run_block(self.refine_16_0, self.refine_16_0_norm, balance_losses, b5[4])
        x = self.d0_norm(self.middle(x))
        x = self._run_block(self.refine_16_1, self.refine_16_1_norm, balance_losses, x, b5[4])
        x = self._run_block(self.d1, self.d1_norm, balance_losses, x, b1[4])
        x = self._run_block(self.refine_32, self.refine_32_norm, balance_losses, x, b5[3])
        x = self._run_block(self.d2, self.d2_norm, balance_losses, x, b1[3])
        x = self._run_block(self.refine_64, self.refine_64_norm, balance_losses, x, b5[2])
        x = self._run_block(self.d3, self.d3_norm, balance_losses, x, b1[2])
        x = self._run_block(self.refine_40, self.refine_40_norm, balance_losses, x, b5[1])
        x = self._run_block(self.d4, self.d4_norm, balance_losses, x, b1[1])
        x = self._run_block(self.refine_24, self.refine_24_norm, balance_losses, x, b5[0])
        x = self._run_block(self.d5, self.d5_norm, balance_losses, x, b1[0])

        s = self.stem(initial_x)
        h = self.fuse(torch.cat((x, s), dim=1))
        if self.fuse_mhc is not None:
            h = h + self.fuse_mhc(x, s)
        h = self.head(h)

        packed = self.to_rgb(h)            # RGB: [B, 12, H, W]
        correction = self.shuffle(packed)  # RGB: [B, 3, 2H, 2W]
        if self.output_mhc is not None:
            logits = self.output_mhc(initial_x, packed)   # [B, 4, H, W]
            gain = 1 + self.shuffle(logits).tanh()        # [B, 1, 2H, 2W]
            correction = correction * gain                # same gain for R, G, B
        base = F.interpolate(initial_x, size=correction.shape[-2:],
                             mode='bicubic', align_corners=False)
        return base + correction, torch.stack(balance_losses).mean()

In [73]:
unet_m = UNet().to(device)
# unet_m.profiling = False          # default; fast path, no syncs, no prints
# unet_m.profiling = True           # prints the per-section table each forward

t = torch.rand(3, 3, 128, 128).to('cuda:0')   # rand, not randn, for realistic ranges
o, _ = unet_m(t)
print(o.shape)                                 # [3, 3, 256, 256]

torch.Size([3, 3, 256, 256])


In [49]:
import torch.nn as nn

def n_params(module: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters (optionally only those with requires_grad=True)."""
    return sum(p.numel() for p in module.parameters()
               if p.requires_grad or not trainable_only)


def param_breakdown(model: nn.Module) -> None:
    """Per-child parameter table — handy for spotting the expensive submodules."""
    total = train = 0
    print(f"{'submodule':24s} {'params':>12s} {'trainable':>12s}")
    for name, child in model.named_children():
        n = sum(p.numel() for p in child.parameters())
        t = sum(p.numel() for p in child.parameters() if p.requires_grad)
        if n:
            print(f"{name:24s} {n:>12,} {t:>12,}")
        total += n
        train += t
    print(f"{'TOTAL':24s} {total:>12,} {train:>12,}")
    
    
print(f"total={n_params(unet_m):,}  trainable={n_params(unet_m, True):,}")
param_breakdown(unet_m)

total=100,038,011  trainable=66,648,875
submodule                      params    trainable
context_encoder             6,101,024            0
context_encoder_b5         27,288,112            0
refine_16_0                20,440,512   20,440,512
refine_16_0_norm                1,024        1,024
middle                     28,372,394   28,372,394
d0_norm                         1,024        1,024
refine_16_1                12,797,846   12,797,846
refine_16_1_norm                  640          640
d1                          1,490,353    1,490,353
d1_norm                           352          352
refine_32                   1,129,864    1,129,864
refine_32_norm                    224          224
d2                            199,669      199,669
d2_norm                           128          128
refine_64                     152,254      152,254
refine_64_norm                     80           80
d3                            399,107      399,107
d3_norm                           160     

In [74]:
import torch.nn as nn

def n_params(module: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters (optionally only those with requires_grad=True)."""
    return sum(p.numel() for p in module.parameters()
               if p.requires_grad or not trainable_only)


def param_breakdown(model: nn.Module) -> None:
    """Per-child parameter table — handy for spotting the expensive submodules."""
    total = train = 0
    print(f"{'submodule':24s} {'params':>12s} {'trainable':>12s}")
    for name, child in model.named_children():
        n = sum(p.numel() for p in child.parameters())
        t = sum(p.numel() for p in child.parameters() if p.requires_grad)
        if n:
            print(f"{name:24s} {n:>12,} {t:>12,}")
        total += n
        train += t
    print(f"{'TOTAL':24s} {total:>12,} {train:>12,}")
    
    
print(f"total={n_params(unet_m):,}  trainable={n_params(unet_m, True):,}")
param_breakdown(unet_m)

total=100,038,203  trainable=66,649,067
submodule                      params    trainable
context_encoder             6,101,024            0
context_encoder_b5         27,288,112            0
refine_16_0                20,440,512   20,440,512
refine_16_0_norm                1,024        1,024
middle                     28,372,394   28,372,394
d0_norm                         1,024        1,024
refine_16_1                12,797,846   12,797,846
refine_16_1_norm                  640          640
d1                          1,490,353    1,490,353
d1_norm                           352          352
refine_32                   1,129,864    1,129,864
refine_32_norm                    224          224
d2                            199,669      199,669
d2_norm                           128          128
refine_64                     152,254      152,254
refine_64_norm                     80           80
d3                            399,107      399,107
d3_norm                           160     

In [50]:
for n, (i, param) in enumerate(unet_m.named_parameters()):
    print(n, i, param.shape)

0 context_encoder.features.0.0.weight torch.Size([32, 3, 3, 3])
1 context_encoder.features.0.1.weight torch.Size([32])
2 context_encoder.features.0.1.bias torch.Size([32])
3 context_encoder.features.1.0.block.0.0.weight torch.Size([32, 1, 3, 3])
4 context_encoder.features.1.0.block.0.1.weight torch.Size([32])
5 context_encoder.features.1.0.block.0.1.bias torch.Size([32])
6 context_encoder.features.1.0.block.1.fc1.weight torch.Size([8, 32, 1, 1])
7 context_encoder.features.1.0.block.1.fc1.bias torch.Size([8])
8 context_encoder.features.1.0.block.1.fc2.weight torch.Size([32, 8, 1, 1])
9 context_encoder.features.1.0.block.1.fc2.bias torch.Size([32])
10 context_encoder.features.1.0.block.2.0.weight torch.Size([16, 32, 1, 1])
11 context_encoder.features.1.0.block.2.1.weight torch.Size([16])
12 context_encoder.features.1.0.block.2.1.bias torch.Size([16])
13 context_encoder.features.1.1.block.0.0.weight torch.Size([16, 1, 3, 3])
14 context_encoder.features.1.1.block.0.1.weight torch.Size([16]

In [37]:
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F


class FeatureLoss(nn.Module):
    """
    Perceptual + frequency + gradient + gram + charbonnier loss, for use on the
    RECONSTRUCTED image from decoder_flow_loss (NOT on the velocity field).

    Key changes vs. the original:
      - B7 input is now ImageNet-normalized. The backbone was trained on
        ImageNet-normalized images; feeding raw [0,1] (or raw decoder output)
        gave wrong perceptual features. Normalization happens ONLY for the
        backbone pass; the pixel/freq/grad/gram terms stay on the plain image.
      - inputs are clamped to [0,1] for safety (the flow decoder can overshoot).
      - the scalar term weights (perceptual/freq/grad/gram/pixel) are now
        constructor args so you can stop perceptual from swamping the velocity
        MSE in decoder_flow_loss. Defaults match the original values.
      - forward() returns (total, parts_dict) -- a dict for clean logging.

    Expects pred/target in [0,1] image range. If your pipeline is in another
    range, convert to [0,1] before calling.
    """

    def __init__(self, weights, device='cuda:0',
                 color_scale=1, perceptual_scale=1, freq_scale=100.0, grad_scale=200.0,
                 gram_scale=500.0, pixel_scale=200.0, face_scale=0, identity_scale=0):
        super().__init__()
        
        # In FeatureLoss.__init__
        self.register_buffer('imagenet_mean', torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1))
        self.register_buffer('imagenet_std', torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1))
        
        from torchvision.models import efficientnet_b7, EfficientNet_B7_Weights
        self.backbone = efficientnet_b7(weights=EfficientNet_B7_Weights.DEFAULT).to(device).eval()
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.layer_names = [
            'features.1.1',
            'features.3.3',
            'features.5.9',
            'features.7.3',
        ]
#         self.layer_names = [
#             'features.4.9.stochastic_depth',
#             'features.5.9.stochastic_depth',
#             'features.6.12.stochastic_depth',
#             'features.7.3.stochastic_depth',
#         ]
        self.features = OrderedDict()
        self.hooks = []
        for name, module in self.backbone.named_modules():
            if name in self.layer_names:
                self.hooks.append(module.register_forward_hook(self._make_hook(name)))


        from facenet_pytorch import InceptionResnetV1

        self.face = InceptionResnetV1(pretrained='vggface2').to(device).eval()
        for p in self.face.parameters():
            p.requires_grad = False

        self.face_layers = ['conv2d_4b', 'repeat_1', 'repeat_2', 'block8']
        self.face_feats = OrderedDict()
        for name, module in self.face.named_modules():
            if name in self.face_layers:
                self.hooks.append(module.register_forward_hook(self._make_face_hook(name)))

        self.face_weights   = [0.3, 0.3, 0.2, 0.2]   # per-layer
        self.face_scale     = 1.0                     # feature-map term
        self.identity_scale = 5.0                     # cosine embedding term   
        self.weights = weights                      # per-layer perceptual weights
        self.perceptual_scale = perceptual_scale
        self.freq_scale = freq_scale
        self.grad_scale = grad_scale
        self.gram_scale = gram_scale
        self.pixel_scale = pixel_scale
        self.color_scale = color_scale
        self.face_scale = face_scale
        self.identity_scale = identity_scale

    def _make_face_hook(self, name): 
        def hook(module, inp, out):
            self.face_feats[name] = out
        return hook

    def _prep_for_face(self, x):
        # ImageNet-normalized -> [0,1] -> [-1,1] -> 160x160.  No clamps.
        x01 = x * self.imagenet_std + self.imagenet_mean
        x = (x01 - 0.5) / 0.5
        return F.interpolate(x, size=(160, 160), mode='bilinear', align_corners=False)

        
    def _make_hook(self, name):
        def hook(module, inp, out):
            self.features[name] = out
        return hook


    @staticmethod
    def gram_matrix(x):
        b, c, h, w = x.size()
        x = x.view(b, c, -1)
        return torch.bmm(x, x.transpose(1, 2)) / (c * h * w)
    
    def color_loss(self, pred, target, pool=16):
        # low-frequency color match: average away all texture, compare color fields.
        pb = F.avg_pool2d(pred.float(),  pool)
        tb = F.avg_pool2d(target.float(), pool)
        return F.l1_loss(pb, tb)


    def charbonnier(self, pred, target, eps=1e-6):
        return torch.mean(torch.sqrt((pred.float() - target.float()) ** 2 + eps))

    def frequency_loss(self, pred, target, alpha=1.0):
        """Focal Frequency Loss: penalizes the frequencies (edges/textures) where
        the prediction is failing hardest. This is the term that most directly
        fights grain/speckle."""
        pred_f32 = pred.float()
        target_f32 = target.float()

        fft_pred = torch.fft.fft2(pred_f32, dim=(-2, -1), norm="ortho")
        fft_target = torch.fft.fft2(target_f32, dim=(-2, -1), norm="ortho")

        diff = fft_pred - fft_target
        distance = torch.abs(diff) + 1e-7

        with torch.no_grad():
            max_distance = distance.amax(dim=(-2, -1), keepdim=True)
            weight = (distance / (max_distance + 1e-7)) ** alpha

        loss = weight * (distance ** 2)
        return loss.mean()

    def gradient_loss(self, pred, target):
        """Matches edge sharpness -- penalizes the high-frequency mismatch that
        shows up as grain."""
        pred_dy = torch.abs(pred[:, :, 1:, :] - pred[:, :, :-1, :])
        pred_dx = torch.abs(pred[:, :, :, 1:] - pred[:, :, :, :-1])
        target_dy = torch.abs(target[:, :, 1:, :] - target[:, :, :-1, :])
        target_dx = torch.abs(target[:, :, :, 1:] - target[:, :, :, :-1])
        return F.l1_loss(pred_dy, target_dy) + F.l1_loss(pred_dx, target_dx)

    def forward(self, pred, target):
        # --- perceptual (backbone) terms: ImageNet-normalized input ---
#         pred_b7 = (pred - self.imagenet_mean) / self.imagenet_std
#         target_b7 = (target - self.imagenet_mean) / self.imagenet_std

        self.features.clear()
        with torch.no_grad():
            self.backbone(target)
            tgt_feats = [self.features[n].clone() for n in self.layer_names]

        self.features.clear()
        self.backbone(pred)
        pred_feats = [self.features[n] for n in self.layer_names]

        with torch.amp.autocast('cuda', enabled=False):

            feature_losses = [F.mse_loss(pred.float(), target.float()) * w
                for pred, target, w in zip(pred_feats, tgt_feats, self.weights)]
#             gram_loss = [F.mse_loss(self.gram_matrix(pred.float()), self.gram_matrix(target.float())) * w
#                 for pred, target, w in zip(pred_feats, tgt_feats, self.weights)]

            gram_loss = [
                F.l1_loss(self.gram_matrix(f_in.float()), self.gram_matrix(f_out.float())) \
                * (w**2) * 1e2 for f_in, f_out, w in zip(pred_feats, tgt_feats, self.weights)]

    

        face_loss = pred.new_zeros((), dtype=torch.float32)
        identity_loss = pred.new_zeros((), dtype=torch.float32)

        if self.face_scale != 0 or self.identity_scale != 0:
            pred_norm = self._prep_for_face(pred)
            target_norm = self._prep_for_face(target)

            self.face_feats.clear()
            with torch.no_grad():
                emb_t = self.face(target_norm)
                tgt_face = [self.face_feats[n].clone() for n in self.face_layers]

            self.face_feats.clear()
            emb_p = self.face(pred_norm)
            pred_face = [self.face_feats[n] for n in self.face_layers]

            if self.face_scale != 0:
                face_losses = [F.mse_loss(a.float(), b.float()) * w
                               for a, b, w in zip(pred_face, tgt_face, self.face_weights)]
                face_loss = sum(face_losses) * self.face_scale

            if self.identity_scale != 0:
                identity_loss = (1.0 - F.cosine_similarity(
                    emb_p.float(), emb_t.float(), dim=1)).mean() * self.identity_scale

        total_face_loss = face_loss + identity_loss

        
        perceptual = sum(feature_losses) * self.perceptual_scale
        gram_losses = sum(gram_loss) * self.gram_scale
#         color_loss = self.color_loss(pred, target) * self.color_scale

        pred_image = pred * self.imagenet_std + self.imagenet_mean
        target_image = target * self.imagenet_std + self.imagenet_mean

        pixel_loss = self.charbonnier(pred_image, target_image) * self.pixel_scale
        freq_loss = self.frequency_loss(pred_image, target_image) * self.freq_scale
        grad_loss = self.gradient_loss(pred_image, target_image) * self.grad_scale


#         total = perceptual + freq_loss + grad_loss + pixel_loss + gram_losses + color_loss + total_face_loss
        total = perceptual + pixel_loss + gram_losses + total_face_loss + freq_loss + grad_loss 

        parts = {
#             "color": color_loss.detach().item(),
            "perceptual": perceptual.detach().item(),
            "gram": [f.detach().item() * self.gram_scale for f in gram_loss],
            "freq": freq_loss.detach().item(),
            "grad": grad_loss.detach().item(),
            "pixel": pixel_loss.detach().item(),
            "feature_losses": [f.detach().item() * self.perceptual_scale for f in feature_losses],
            "face": face_loss.detach().item(),
            "identity": identity_loss.detach().item()
        }
        return total, parts

    def __del__(self):
        for h in self.hooks:
            try:
                h.remove()
            except Exception:
                pass

In [ ]:
# import math
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# def timestep_embedding(t, dim, max_period=10000):
#     half = dim // 2
#     freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
#     args = (t.float() * 1000.0)[:, None] * freqs[None]      # t in [0,1] -> scale by 1000
#     emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
#     if dim % 2:
#         emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
#     return emb

# class FlowDecoder(nn.Module):
#     """
#     Your UNet, turned into a velocity field in PIXEL space.
#       source  = a start image (noise, or upsampled blurry input)
#       x_t     = (1-t)*source + t*target          (the path point)
#       target  = the sharp image y
#       output  = velocity (target - source), conditioned on the latents
#     The latents (skips, deep_skip) condition every step. x_t is the moving input.
#     """
#     def __init__(self, base_unet, hidden_dim=320):
#         super().__init__()
#         self.unet = base_unet                                  # your UNet, unchanged
#         # turn the timestep into a per-channel shift on the deep feature (FiLM-style)
#         self.t_mlp = nn.Sequential(
#             nn.Linear(256, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim),
#         )

#     def forward(self, skips, deep_skip, x_t, t):
#         # inject time into the deepest conditioning feature (simple, effective)
#         temb = self.t_mlp(timestep_embedding(t, 256))          # (B, 320)
#         deep_skip[0] = deep_skip[0] + temb[:, :, None, None]         # broadcast shift
#         # x_t is the moving image the decoder refines; pass it where `x` used to go
#         return self.unet(skips, deep_skip, x_t)                # outputs velocity image

In [ ]:
# def make_conditioning(context_encoder, flow, x, jitter=True):
#     context_feats = context_encoder(x)
#     fixed_context = [f.contiguous() for f in context_feats][:-1]
#     cn  = flow._norm(fixed_context)
#     src = [c + flow.sigma_spice * torch.randn_like(c) for c in cn] if jitter else cn
#     t0  = torch.zeros(src[0].shape[0], device=x.device)
#     pred_latents = flow._denorm(flow.velocity_model(src, t0, cn))
#     return pred_latents[::-1], context_feats[::-1]   # skips, deep_skip

# def sigma_of(t, sigma_max=2.5):
#     # t in [0,1]: t=0 -> max blur, t=1 -> sharp (sigma -> 0)
#     return sigma_max * (1.0 - t)

# def blur_y(y, sigma):
#     # per-sample Gaussian blur at a given sigma (batch=1 path; see caveat for batched)
#     s = float(sigma)
#     if s < 1e-3:
#         return y                      # ~0 sigma => sharp y, no-op (also guards negatives)
#     k = max(3, int(y.shape[-1] * 0.03)); k = k if k % 2 == 1 else k + 1
#     return TF.gaussian_blur(y, kernel_size=[k, k], sigma=[s, s])


# def decoder_flow_loss(flow_decoder, skips, deep_skip, target_img, blurry_img,
#                       feature_loss, sigma_max=3, eps=1e-2,
#                       velocity_weight=1.0, perceptual_weight=0.5,
#                       noise=False, noise_floor=0.12):
#     B = target_img.shape[0]; device = target_img.device
#     t  = torch.rand(B, device=device); t_ = t.view(-1, 1, 1, 1)

#     # finite-difference window, clamped to [0,1] so it stays valid at the ends
#     t_hi = (t + eps).clamp(max=1.0)
#     t_lo = (t - eps).clamp(min=0.0)

#     sig    = sigma_of(t)
#     sig_hi = sigma_of(t_hi)
#     sig_lo = sigma_of(t_lo)

#     # ---- INPUT: a genuine blurred y at level t (on-manifold) ----
#     x_t  = torch.stack([blur_y(target_img[i:i+1], sig[i])[0]    for i in range(B)])

#     # ---- TARGET: true path derivative from CLEAN blurs (noise must NOT enter here) ----
#     x_hi = torch.stack([blur_y(target_img[i:i+1], sig_hi[i])[0] for i in range(B)])
#     x_lo = torch.stack([blur_y(target_img[i:i+1], sig_lo[i])[0] for i in range(B)])
#     dt_  = (t_hi - t_lo).view(-1, 1, 1, 1)
#     v_target = (x_hi - x_lo) / dt_

#     # ---- optional noise on the INPUT ONLY (robustness / breaks blur null-space) ----
#     if noise:
#         x_t = x_t + noise_floor * torch.randn_like(x_t)

#     # ---- decoder sees (possibly noisy) x_t, predicts the clean velocity ----
#     v_pred = flow_decoder(skips, deep_skip, x_t, t)
#     velocity_loss = F.mse_loss(v_pred.float(), v_target.float())

#     pred_img = (x_t + (1 - t_) * v_pred).clamp(0, 1)   # rough proxy on a curved path; ok as t->1
#     feat_total, parts = feature_loss(pred_img, target_img)
#     parts['vel loss'] = (velocity_weight * velocity_loss).item()
#     return velocity_weight * velocity_loss + perceptual_weight * feat_total, parts


# @torch.no_grad()
# def decoder_sample(flow_decoder, skips, deep_skip, blurry_img, target_shape, device,
#                    steps=4, sigma_max=3, noise=False, noise_floor=0.02):
#     # start from the real blurred input upsampled to target size (treated as t=0 = max blur)
#     x = blurry_img

#     dt = 1.0 / steps
#     for j in range(steps):
#         t = torch.full((x.shape[0],), j * dt, device=device)
#         if j % 1 == 0 and j < steps - 1:
#             inp = torch.cat([x, blurry_img], dim=1)
#         else:
#             inp = torch.cat([x, x], dim=1)
#         v = flow_decoder(skips, deep_skip, inp, t)
#         x = x + v * dt                                   # Euler step up the blur path
#         if noise and j < steps - 1:                      # never add noise on the final step
#             remaining = 1.0 - (j + 1) * dt               # fade noise out as we approach sharp
#             x = x + noise_floor * remaining * torch.randn_like(x)
#     return x

# def sample_train(flow_decoder, skips, deep_skip, blurry_img, target_shape, steps=4):
#     x = blurry_img
#     dt = 1.0 / steps
#     for j in range(steps):
#         t = torch.full((x.shape[0],), j * dt, device=x.device)
#         if j % 1 == 0 and j < steps - 1:
#             inp = torch.cat([x, blurry_img], dim=1)
#         else:
#             inp = torch.cat([x, x], dim=1)
#         v = flow_decoder(skips, deep_skip, inp, t)
#         x = x + v * dt
#     return x

In [38]:
# ============================================================================
#  DEBUG SAVE  (flow-decoder, few-step blurry-start walk, NO sampling noise)
#  Matches the new training path: same step count, noise off.
# ============================================================================
def save_debug_triplet(flow_decoder, y_folder, filenames,
                       counter, epoch, save_dir="save_image_folder", num_samples=2,
                       x_size=512, y_size=1024, pretrain=False, steps=4):
    os.makedirs(save_dir, exist_ok=True)
    unet_m.eval()

    with torch.no_grad():
        sampled_fnames = random.sample(filenames, min(num_samples, len(filenames)))
        print(sampled_fnames)
        for idx, fname in enumerate(sampled_fnames):
            img = Image.open(os.path.join(y_folder, fname)).convert("RGB")
            y_img = TF.resize(img, (y_size, y_size), interpolation=InterpolationMode.BICUBIC)
            x_img = TF.resize(img, (x_size, x_size), interpolation=InterpolationMode.BICUBIC)
            x_t  = transform_x(x_img).unsqueeze(0).cuda()       # blurry input -> encoder
            x    = transform_clean(x_img).unsqueeze(0).cuda()
            y_t  = transform_clean(y_img).unsqueeze(0).cuda()

            with autocast('cuda'):
                y_hat, _ = unet_m(x_t)

            tag = f"{counter}_{epoch}_{idx}_{uuid.uuid4().hex[:6]}_{os.path.splitext(fname)[0]}"
            for suffix, t in [("x", x), ("x_blur", x_t), ("yhat", y_hat), ("y", y_t)]:
                tensor_to_image(denormalize(t[0].detach())).save(
                    os.path.join(save_dir, f"{tag}_{suffix}.png"))
            print(f"  Saved: {tag}")
    unet_m.train()         # Call it on your model.

In [ ]:
# model_save_path = 'post_training_test_10_30.pth'

In [19]:

model_save_path = 'new_training_11_2.pth'

In [75]:
unet_m.load_state_dict(torch.load(model_save_path, map_location=device), strict=False)

<All keys matched successfully>

In [ ]:
model_save_path

In [39]:
def worst_patch_loss(pred, target, patch_size=16, fraction=0.25):
    # Your training tensors are ImageNet-normalized.
    # Multiplying differences by std converts errors to ordinary RGB units.
    std = pred.new_tensor(
        [0.229, 0.224, 0.225], dtype=torch.float32
    ).view(1, 3, 1, 1)

    error = ((pred.float() - target.float()) * std).abs()
    error = error.mean(dim=1, keepdim=True)

    size = min(patch_size, error.shape[-2], error.shape[-1])
    patch_errors = F.avg_pool2d(
        error, kernel_size=size, stride=size,
        ceil_mode=True, count_include_pad=False
    ).flatten(1)

    count = max(1, int(patch_errors.shape[1] * fraction))
    return patch_errors.topk(count, dim=1).values.mean()

In [ ]:
import os
import random
import torch
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR


model_save_path = 'new_training_11_2.pth'



y_folder = "E:/close_alexa/3y_1024_1024/"
# y_folder = "E:/close_alexa/3y_1024_1024_quartered/"
total_steps = len(os.listdir(y_folder))
all_filenames = get_filenames_from_y(y_folder)
train_filenames, val_filenames = split_filenames(all_filenames)


# Pretraining
# feat_loss = FeatureLoss(weights=[0.4, 0.12, 0.12, 0.3], device='cuda:0',
#                  color_scale=0, perceptual_scale=0, freq_scale=0, grad_scale=0,
#                  gram_scale=10, pixel_scale=60, face_scale=0, identity_scale=0).cuda()

# Mid-training — provisional starting values
feat_loss = FeatureLoss(weights=[0.4, 0.1, 0.12, 0.3], device='cuda:0',
                 color_scale=0, perceptual_scale=1, freq_scale=0, grad_scale=0,
                 gram_scale=15, pixel_scale=20, face_scale=0, identity_scale=0).cuda()

# # # Final fine-tuning — provisional starting values
# feat_loss = FeatureLoss(weights=[0.4, 0.07, 0.12, 0.3], device='cuda:0',
#                  color_scale=0, perceptual_scale=1, freq_scale=100, grad_scale=10,
#                  gram_scale=0, pixel_scale=0, face_scale=1000, identity_scale=1).cuda()


scaler = GradScaler("cuda")

batch_size = 1
unet_lr    = 1e-5    # the flow decoder (unet_m)

unet_optimizer = torch.optim.AdamW(unet_m.parameters(),  lr=unet_lr, weight_decay=1e-4)
unet_scheduler = CosineAnnealingLR(unet_optimizer, T_max=total_steps, eta_min=1e-6)

factor = 12
step = 0
PRETRAIN_MODE = True
num_epochs = 10 if PRETRAIN_MODE else 40
latent_loss_history = deque(maxlen=1000)

latent_loss_average_tally = deque(maxlen=400)

noise_floor=0.02
noise=False

for epoch in range(1, num_epochs + 1):
    print(f"\n{'='*60}\nEpoch {epoch}/{num_epochs}\n{'='*60}")
    random.shuffle(train_filenames)
    unet_m.train()
    counter = 0

    if PRETRAIN_MODE:
        train_loader = load_pretrain_images(y_folder, train_filenames, batch_size,
                                            x_size=16*factor, y_size=32*factor, augment=True)
    else:
        train_loader = load_images_from_filenames(y_folder, train_filenames, batch_size, augment=True)

    for x_batch, y_batch in train_loader:
        step += 1
        counter += 1

        with torch.autocast("cuda"):
            y_hat, balance_loss = unet_m(x_batch)
            reconstruction_loss, loss_parts = feat_loss(y_hat, y_batch)
            local_loss = worst_patch_loss(y_hat, y_batch)

            local_weight = 10
            loss = reconstruction_loss + local_weight * local_loss
            loss = loss + 0.01 * balance_loss

            loss_parts["worst_patch"] = local_loss.detach().item() * local_weight

        scaler.scale(loss).backward()   # decoder -> flow -> encoder
        latent_loss_history.append(loss.item())
        scaler.step(unet_optimizer)     # decoder
        scaler.update()
        unet_scheduler.step()
        unet_optimizer.zero_grad(set_to_none=True)

        # ==========================================
        # LOGGING & SAVING
        # ==========================================
        if step % 20 == 0:
            latent_loss_average = sum(latent_loss_history) / len(latent_loss_history)
            latent_loss_average_tally.append(latent_loss_average)
            
            print(
                f"loss_parts {loss_parts} | "
                f"Avg={latent_loss_average:.6f}"
            )

        if step % 120 == 0 and step > 0:
            torch.save(unet_m.state_dict(), model_save_path)
            print("flow and U-Net saved")
            
        if step % 1500 == 0:
            try:
                from IPython.display import clear_output
                clear_output(wait=True)
            except ImportError:
                pass
            
        if step % 3364 == 0:
            if PRETRAIN_MODE:
                save_debug_triplet(unet_m, y_folder, train_filenames, counter, epoch,
                   x_size=16*factor, y_size=32*factor, pretrain=True)
            else:
                save_debug_triplet(model, y_folder, train_filenames, counter, epoch)


Epoch 1/10


In [ ]:
save_debug_triplet(unet_m, y_folder, train_filenames, counter, epoch,
   x_size=16*factor*2, y_size=32*factor, pretrain=True, steps=4)

In [52]:
model_save_path

'new_training_11_2.pth'

In [69]:
torch.save(unet_m.state_dict(), model_save_path)

In [ ]:
cpu_state = {
    name: tensor.detach().cpu()
    for name, tensor in unet_m.state_dict().items()
}

torch.save(cpu_state, "unet_oom_recovery.pth")
print("Model saved from CPU.")

In [ ]:
model_save_path

In [ ]:
# model_save_path = 'post_training_test_10_30.pth'


unet_m.load_state_dict(torch.load(model_save_path, map_location=device), strict=False)


In [ ]:
def crop_4k_bottom(x):
    H, W = x.shape[-2:]
    assert H >= 2160 and W >= 3840, f"model output is {H}x{W}, need ≥ 2160x3840"
    return x[..., H - 2160:H, W // 2 - 1920 : W // 2 + 1920]

In [ ]:
import os
import time
import torch
import torch.nn.functional as F
from PIL import Image

def pad_to_32(x):
    h, w = x.shape[-2:]
    pad_h = (-h) % 32
    pad_w = (-w) % 32
    return F.pad(x, (0, pad_w, pad_h, 0), mode="reflect")

path_1080 = 'E:/close_alexa/sample_upscale_2026/'
out_path  = 'E:/close_alexa/sample_upscale_2026_out/'
os.makedirs(out_path, exist_ok=True)
files = os.listdir(path_1080)[80:]
unet_m.eval()

for filename in files:
    if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp")):
        continue

    with Image.open(os.path.join(path_1080, filename)) as img:
        img = img.convert("RGB")
        img_tensor = transform_clean(img).unsqueeze(0)

    h, w = img_tensor.shape[-2:]
    img_tensor = pad_to_32(img_tensor).to("cuda:0")

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.float16):
            output, _ = unet_m(img_tensor)

        # Remove the doubled top/right padding from this 2× model.
        output = output[:, :, -2*h:, :2*w].contiguous()

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        output = output.float().cpu()

    print(f"{filename}: {elapsed:.2f}s | output: {tuple(output.shape)}")
    output_img = denormalize(output[0])
    tensor_to_image(output_img).save(os.path.join(out_path, f'401_a{filename}'))
    del output, output_img, img_tensor